In [1]:
import pandas as pd
chemin = "./data/raw/comptages.csv"
df = pd.read_csv(chemin, sep=";")
df["t_1h"] = pd.to_datetime(df["t_1h"])
df.head(5)

,iu_ac,libelle,t_1h,q,k,etat_trafic,iu_nd_amont,libelle_nd_amont,iu_nd_aval,libelle_nd_aval,etat_barre
0,1960,Av_de_l'Opera,2026-08-31 23:00:00+00:00,156.0,1.74667,Fluide,2139,Pl_de_l'Opera,211,Av_de_l'Opera-Antin-Augustin,Ouvert
1,57,St_Jacques,2026-08-31 23:00:00+00:00,206.0,3.22778,Fluide,52,St_Jacques-Galande,53,Bd_St_Germain-St_Jacques,Ouvert
2,298,Av_de_l'Opera,2026-08-31 23:00:00+00:00,257.0,1.73667,Fluide,178,Av_de_l'Opera-Echelle,87,Pl_Andre_Malraux,Ouvert
3,300,Av_de_l'Opera,2026-08-31 23:00:00+00:00,158.0,2.21722,Fluide,177,Av_de_l'Opera-Pyramides,178,Av_de_l'Opera-Echelle,Ouvert
4,367,Av_de_l'Opera,2026-08-31 23:00:00+00:00,156.0,1.44167,Fluide,211,Av_de_l'Opera-Antin-Augustin,179,Av_de_l'Opera-Petits_Champs,Ouvert


**1 - Variables temporelles en heure de Paris**

In [2]:
df = df.rename(columns={"t_1h": "t_paris"})

df["heure"] = df["t_paris"].dt.hour
df["mois"] = df["t_paris"].dt.month

jours_fr = {0: "Lundi", 1: "Mardi", 2: "Mercredi", 3: "Jeudi", 4: "Vendredi", 5: "Samedi", 6: "Dimanche"}
df["jour_semaine"] = df["t_paris"].dt.dayofweek.map(jours_fr)

df["weekend"] = df["t_paris"].dt.dayofweek >= 5

df[["t_paris", "heure", "jour_semaine", "weekend", "mois"]].head()

,t_paris,heure,jour_semaine,weekend,mois
0,2026-08-31 23:00:00+00:00,23,Lundi,False,8
1,2026-08-31 23:00:00+00:00,23,Lundi,False,8
2,2026-08-31 23:00:00+00:00,23,Lundi,False,8
3,2026-08-31 23:00:00+00:00,23,Lundi,False,8
4,2026-08-31 23:00:00+00:00,23,Lundi,False,8


J'ai renommé t_1h en t_paris, puis extrait quatre nouvelles colonnes à partir de cette date (déjà en heure de Paris) : heure (0-23), mois (1-12), jour_semaine (nom du jour en français) et weekend (booléen, vrai pour samedi/dimanche).

**2 - Encodage : montrez que l'ordre existe (ou non) de choisir ordinal ou one-hot**

In [3]:
print(df["etat_trafic"].unique())
print(df["etat_barre"].unique())

<StringArray>
['Fluide', 'Inconnu', 'Pré-saturé', 'Saturé', 'Bloqué']
Length: 5, dtype: str
<StringArray>
['Ouvert', 'Invalide', 'Barré']
Length: 3, dtype: str


In [4]:
df.groupby("etat_trafic")["q"].mean().sort_values()

etat_trafic
Inconnu       257.899923
Fluide        268.837757
Pré-saturé    546.233831
Saturé        592.074020
Bloqué        725.132450
Name: q, dtype: float64

In [5]:
df.groupby("etat_barre")["q"].mean().sort_values()

etat_barre
Barré       110.654321
Ouvert      287.529467
Invalide           NaN
Name: q, dtype: float64

In [6]:
ordre_trafic = ["Fluide", "Pré-saturé", "Saturé", "Bloqué"]
df["etat_trafic_ord"] = df["etat_trafic"].map({etat: i for i, etat in enumerate(ordre_trafic)})


In [7]:
df = pd.get_dummies(df, columns=["etat_barre"], prefix="barre")

In [8]:
df.filter(like="barre").columns
df[["barre_Ouvert", "barre_Invalide", "barre_Barré"]].drop_duplicates()


,barre_Ouvert,barre_Invalide,barre_Barré
0,True,False,False
7,False,True,False
54693,False,False,True


In [9]:
df.head()

,iu_ac,libelle,t_paris,q,k,etat_trafic,iu_nd_amont,libelle_nd_amont,iu_nd_aval,libelle_nd_aval,heure,mois,jour_semaine,weekend,etat_trafic_ord,barre_Barré,barre_Invalide,barre_Ouvert
0,1960,Av_de_l'Opera,2026-08-31 23:00:00+00:00,156.0,1.74667,Fluide,2139,Pl_de_l'Opera,211,Av_de_l'Opera-Antin-Augustin,23,8,Lundi,False,0.0,False,False,True
1,57,St_Jacques,2026-08-31 23:00:00+00:00,206.0,3.22778,Fluide,52,St_Jacques-Galande,53,Bd_St_Germain-St_Jacques,23,8,Lundi,False,0.0,False,False,True
2,298,Av_de_l'Opera,2026-08-31 23:00:00+00:00,257.0,1.73667,Fluide,178,Av_de_l'Opera-Echelle,87,Pl_Andre_Malraux,23,8,Lundi,False,0.0,False,False,True
3,300,Av_de_l'Opera,2026-08-31 23:00:00+00:00,158.0,2.21722,Fluide,177,Av_de_l'Opera-Pyramides,178,Av_de_l'Opera-Echelle,23,8,Lundi,False,0.0,False,False,True
4,367,Av_de_l'Opera,2026-08-31 23:00:00+00:00,156.0,1.44167,Fluide,211,Av_de_l'Opera-Antin-Augustin,179,Av_de_l'Opera-Petits_Champs,23,8,Lundi,False,0.0,False,False,True


Avant de choisir une méthode d'encodage, j'ai vérifié s'il existait un ordre naturel dans les variables catégorielles, en comparant le débit moyen (q) par catégorie :

etat_trafic : les moyennes suivent une vraie progression logique (Fluide 269 < Pré-saturé 546 < Saturé 592 < Bloqué 725) donc un ordre existe réellement, donc encodage ordinal (etat_trafic_ord, de 0 à 3). Inconnu ne fait pas partie de cette échelle (ce n'est pas un niveau de trafic mais une absence de mesure) et reste à NaN dans cette colonne. La colonne d'origine etat_trafic que j'ai supprimé.
etat_barre : pas de progression logique entre Ouvert/Barré/Invalide, ce sont des états différents et non des degrés d'un même phénomène donc pas d'ordre naturel, donc encodage one-hot (pd.get_dummies), qui a automatiquement remplacé la colonne d'origine.


**3 - Comparez l'IQR a ce qui est physiquement impossible sur vos capteurs. Décidez : suprimer, plafonner, garder et pourquoi ?**

In [10]:
df[["q", "k"]].describe()

,q,k
count,118464.000000,115069.000000
mean,287.287589,5.325570
std,205.428998,6.703946
min,0.000000,0.000000
25%,113.000000,1.309450
50%,260.000000,3.287220
75%,426.000000,6.225000
max,5473.950000,78.702220


In [11]:
q1, q3 = df["q"].quantile([0.25, 0.75])
iqr_q = q3 - q1
borne_haute_q = q3 + 1.5 * iqr_q
print(f"IQR q = {iqr_q:.1f} (Q1={q1}, Q3={q3})  borne haute = {borne_haute_q:.1f}")

k1, k3 = df["k"].quantile([0.25, 0.75])
iqr_k = k3 - k1
borne_haute_k = k3 + 1.5 * iqr_k
print(f"IQR k = {iqr_k:.1f} (Q1={k1}, Q3={k3})  borne haute = {borne_haute_k:.1f}")

IQR q = 313.0 (Q1=113.0, Q3=426.0)  borne haute = 895.5
IQR k = 4.9 (Q1=1.30945, Q3=6.225)  borne haute = 13.6


In [12]:
print(df["q"].max(), (df["q"] > borne_haute_q).sum(), f"{(df['q'] > borne_haute_q).mean():.1%}")
print(df["k"].max(), (df["k"] > borne_haute_k).sum(), f"{(df['k'] > borne_haute_k).mean():.1%}")

5473.95 725 0.5%
78.70222 10016 7.0%


une partie des valeurs "extrêmes" selon l'IQR est exactement le phénomène qu'on veut étudier, pas une anomalie de capteur. Supprimer ces lignes reviendrait à effacer les embouteillages de l'analyse

In [13]:
incoherent = df[(df["q"] > 0) & (df["k"] == 0)]
print(len(incoherent), "lignes avec débit positif mais occupation nulle")

180 lignes avec débit positif mais occupation nulle


In [14]:
print(len(df[(df["k"] > 0) & (df["q"] == 0)]), "lignes avec occupation positive mais débit nul")

54 lignes avec occupation positive mais débit nul


In [15]:
for seuil in [1500, 2000, 3000, 4000, 5000]:
    print(seuil, (df["q"] > seuil).sum())

1500 1
2000 1
3000 1
4000 1
5000 1


In [16]:
df[df["q"] > 3000][["t_paris", "iu_ac", "libelle", "q", "k"]].sort_values("q", ascending=False).head(20)

,t_paris,iu_ac,libelle,q,k
48757,2026-07-01 10:00:00+00:00,1961,Av_de_l'Opera,5473.95,25.21889


In [17]:
stats_capteur = df.groupby("iu_ac")["q"].agg(["median", "max"])
stats_capteur["ratio"] = stats_capteur["max"] / stats_capteur["median"]
stats_capteur.sort_values("ratio", ascending=False).head(10)

,median,max,ratio
iu_ac,,,
1961,403.0,5473.95,13.583002
5097,158.0,790.00,5.000000
5095,158.0,790.00,5.000000
600,34.0,123.00,3.617647
599,34.0,123.00,3.617647
597,34.0,123.00,3.617647
598,34.0,123.00,3.617647
299,171.0,477.00,2.789474
301,171.0,477.00,2.789474


Aucun capteur ne se détache de façon flagrante, le ratio le plus élevé est 13.6 (capteur 1961, celui qu'on avait déjà repéré avec un max de 5473.95), les autres tournent entre 2.8 et 5. Ce ne sont pas des écarts énormes (on s'attendrait à du 50x-100x pour un vrai bug de capteur), donc ce test-là ne fait remonter aucune valeur à supprimer en plus de ce qu'on a déjà trouvé.

**4 - Statistiques par troncon + 3 phrases d'interprétation**

In [18]:
stats_troncon = df.groupby("libelle").agg(
    q_moyen=("q", "mean"),
    q_median=("q", "median"),
    q_std=("q", "std"),
    k_moyen=("k", "mean"),
    k_median=("k", "median"),
)
stats_troncon.sort_values("q_moyen", ascending=False)

,q_moyen,q_median,q_std,k_moyen,k_median
libelle,,,,,
Bd_Vincent_Auriol,342.335556,340.0,212.905344,6.020265,3.94445
Av_de_l'Opera,266.764558,244.0,180.790057,4.864471,3.52500
St_Jacques,227.942733,159.0,205.308722,4.525985,1.63333


Bd_Vincent_Auriol est le tronçon le plus chargé des trois, avec à la fois le débit moyen le plus élevé (342 véhicules par heures) et le taux d'occupation moyen le plus fort (6%), ce qui en fait l'axe le plus circulé.

St_Jacques a le débit le plus instable : son écart-type (205) est presque aussi grand que sa moyenne (228), ce qui montre une forte alternance entre des heures très calmes et des pics ponctuels de trafic, contrairement à Bd_Vincent_Auriol dont le trafic est plus régulier (écart-type nettement inférieur à la moyenne).

Sur les trois tronçons, la médiane reste toujours en dessous de la moyenne, ce qui confirme que le débit horaire est asymétrique à droite la majorité des heures sont calmes, mais quelques heures de forte affluence tirent la moyenne vers le haut, cohérent avec ce qu'on avait déjà observé avec le boxplot.

In [19]:
lignes_vides = df["q"].isna() & df["k"].isna()
print(lignes_vides.sum(), "lignes sans aucune information (q et k manquants)")

df = df[~lignes_vides].reset_index(drop=True)
print(len(df), "lignes restantes")

12054 lignes sans aucune information (q et k manquants)


131958 lignes restantes


In [20]:
from pathlib import Path

dossier_sortie = Path("data/processed")
dossier_sortie.mkdir(parents=True, exist_ok=True)
df.to_csv(dossier_sortie / "comptages_apres_transformation_statistiques.csv", index=False)
print(f"{len(df)} lignes enregistrées dans {dossier_sortie / 'comptages_apres_transformation_statistiques.csv'}")

131958 lignes enregistrées dans data/processed/comptages_apres_transformation_statistiques.csv
